# QGAIN v4.1.0 finalization — reviewed freeze candidate

This notebook performs the **semantic and governance finalization** of the validated QGAIN v4.0.1 cohort extraction. It does not recompute or alter the four numerical estimators. Instead, it:

- verifies and pins the validated v4.0.1 source artifacts;
- upgrades the feature registry to the final scientific roles;
- quantifies robustness rather than treating audit coverage as success;
- adds participant-balanced sensitivity summaries;
- exposes drift uncertainty and evidence fields;
- creates publication, support, ML-interface, and feature-passport artifacts;
- produces a freeze-ready manifest for an atomic PowerShell freeze.

**Final family claim:** QGAIN quantifies contextual operating level and within-recording level variation in strict-speech regions. The measurements are compatible with recording gain, distance, or automatic level control, but are not source-identifying and can also reflect vocal intensity, prosody, respiration, dysarthria, fatigue, and task structure.

No QGAIN family scalar or standalone accept/reject threshold is constructed.

## 0. Environment, acceptance decision, and paths

In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "notebooks/02_feature_extraction").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing reviewed folders.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from paper1_qc_reviewed.qgain_v410 import (
    ANALYSIS_FEATURES,
    EXPLORATORY_FEATURES,
    MEASUREMENT_VERSION,
    PRIMARY_ANALYSIS_FEATURES,
    SECONDARY_ANALYSIS_FEATURES,
    feature_registry_frame,
    finalize_recording_frame_v410,
    measurement_long_frame,
    model_interface_frame,
)

RUN_PACKAGE_TESTS = True
SUBJECT_BALANCED_ITERATIONS = 1000
SUBJECT_BALANCED_SEED = 20260802
SCIENTIFIC_REVIEW_DECISION = "ACCEPT_QGAIN_V410"
SCIENTIFIC_REVIEWER = "Nevena Musikic"
SCIENTIFIC_REVIEW_RATIONALE = (
    "Feature-specific scientific audit completed after correction of the canonical "
    "strict_speech/primary interval contract. Numerical estimators are retained unchanged; "
    "typical level is contextual, within-segment IQR is primary mixed, between-segment MAD "
    "is secondary mixed, and drift is exploratory/contextual. No scalar or standalone gate is approved."
)

SOURCE_ROOT = PROJECT_ROOT / "outputs/reviewed" / "gain_dynamics" / "qgain-v4.0.1-candidate"
OUTPUT_ROOT = PROJECT_ROOT / "outputs/reviewed" / "gain_dynamics" / "qgain-v4.1.0-candidate"
TABLES = OUTPUT_ROOT / "tables"
VALIDATION = OUTPUT_ROOT / "validation"
FIGURES = OUTPUT_ROOT / "figures"
GALLERIES = OUTPUT_ROOT / "galleries"
AUDIT = OUTPUT_ROOT / "audit"
MANIFESTS = OUTPUT_ROOT / "manifests"
PASSPORTS = OUTPUT_ROOT / "feature_passports"
SOURCE_EVIDENCE = VALIDATION / "source_qgain_v401"
for folder in [TABLES, VALIDATION, FIGURES, GALLERIES, AUDIT, MANIFESTS, PASSPORTS, SOURCE_EVIDENCE]:
    folder.mkdir(parents=True, exist_ok=True)


def save_table(frame: pd.DataFrame, stem: Path) -> None:
    stem.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(stem.with_suffix(".csv"), index=False)
    try:
        frame.to_parquet(stem.with_suffix(".parquet"), index=False)
    except Exception as exc:
        print(f"Parquet not written for {stem.name}: {type(exc).__name__}: {exc}")


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def require_file(relative: str) -> Path:
    path = SOURCE_ROOT / relative
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def read_table(relative_stem: str) -> pd.DataFrame:
    stem = SOURCE_ROOT / relative_stem
    for suffix in [".csv", ".parquet"]:
        path = stem.with_suffix(suffix)
        if path.exists():
            try:
                return pd.read_csv(path) if suffix == ".csv" else pd.read_parquet(path)
            except pd.errors.EmptyDataError:
                return pd.DataFrame()
            except ImportError:
                continue
    raise FileNotFoundError(stem)


print({
    "project_root": str(PROJECT_ROOT),
    "source_root": str(SOURCE_ROOT),
    "output_root": str(OUTPUT_ROOT),
    "measurement_version": MEASUREMENT_VERSION,
    "scientific_review_decision": SCIENTIFIC_REVIEW_DECISION,
})

## 1. G1 — pinned source provenance and final feature contract

In [ ]:
source_manifest_path = require_file("manifests/qgain_v401_candidate_manifest.json")
source_manifest = json.loads(source_manifest_path.read_text(encoding="utf-8"))

required_source_artifacts = {
    "recording_features_csv": require_file("tables/qgain_v401_recording_features.csv"),
    "frame_ledger_parquet": require_file("tables/qgain_v401_frame_ledger.parquet"),
    "segment_ledger_parquet": require_file("tables/qgain_v401_segment_ledger.parquet"),
    "empirical_summary_csv": require_file("tables/qgain_v401_empirical_summary.csv"),
    "segment_deletion_csv": require_file("validation/qgain_v401_segment_delete_one_long.csv"),
    "boundary_sensitivity_csv": require_file("validation/qgain_v401_boundary_guard_sensitivity.csv"),
    "persistence_csv": require_file("validation/qgain_v401_repeated_recording_persistence.csv"),
    "drift_permutation_csv": require_file("validation/qgain_v401_drift_time_order_permutation.csv"),
    "canonical_contract_csv": require_file("validation/qgain_v401_canonical_interval_contract.csv"),
    "source_manifest_json": source_manifest_path,
}
source_inventory = pd.DataFrame([
    {"artifact": name, "path": str(path), "bytes": path.stat().st_size, "sha256": sha256(path)}
    for name, path in required_source_artifacts.items()
])
save_table(source_inventory, VALIDATION / "qgain_v410_source_artifact_inventory")

registry = feature_registry_frame()
save_table(registry, TABLES / "qgain_v410_feature_registry")

canonical_contract = pd.read_csv(required_source_artifacts["canonical_contract_csv"])
source_extraction_errors = read_table("audit/qgain_v401_extraction_errors")
source_robustness_errors = read_table("audit/qgain_v401_robustness_errors")
source_gallery_errors = read_table("audit/qgain_gallery_errors")

role_map = registry.set_index("feature")["analysis_priority"].to_dict()
g1 = pd.DataFrame([
    {"gate":"G1", "check":"validated v4.0.1 cohort extraction completed", "passed":bool(source_manifest.get("cohort_extraction_completed"))},
    {"gate":"G1", "check":"source cohort used strict_speech/primary only", "passed":canonical_contract["view"].astype(str).eq("strict_speech").all() and canonical_contract["profile"].astype(str).eq("primary").all()},
    {"gate":"G1", "check":"source extraction, robustness, and gallery error tables are empty", "passed":source_extraction_errors.empty and source_robustness_errors.empty and source_gallery_errors.empty},
    {"gate":"G1", "check":"exactly four retained non-gating features", "passed":len(registry) == 4 and not registry["standalone_gate_allowed"].any()},
    {"gate":"G1", "check":"final feature hierarchy is explicit", "passed":role_map == {
        "qgain_typical_speech_level_dbfs":"primary_context",
        "qgain_within_segment_iqr_db":"primary",
        "qgain_between_segment_mad_db":"secondary",
        "qgain_abs_drift_db_per_min":"exploratory",
    }},
    {"gate":"G1", "check":"family scalar remains prohibited", "passed":registry["composite_use_prohibited"].all()},
])
save_table(g1, VALIDATION / "qgain_v410_g1_checks")
display(registry)
display(g1)

## 2. G2 — exact numerical continuity and finalized exports

In [ ]:
source_recordings = read_table("tables/qgain_v401_recording_features")
recording_table = finalize_recording_frame_v410(source_recordings)

# The finalization is semantic only: all four numerical values must remain bitwise equal.
equivalence_rows = []
for feature in ANALYSIS_FEATURES:
    left = pd.to_numeric(source_recordings[feature], errors="coerce")
    right = pd.to_numeric(recording_table[feature], errors="coerce")
    equal = np.array_equal(left.to_numpy(), right.to_numpy(), equal_nan=True)
    delta = (right - left).abs()
    equivalence_rows.append({
        "feature": feature,
        "bitwise_equal_with_nan": bool(equal),
        "n_compared": int((left.notna() & right.notna()).sum()),
        "maximum_absolute_delta": float(delta.max()) if delta.notna().any() else 0.0,
    })
equivalence = pd.DataFrame(equivalence_rows)
save_table(equivalence, VALIDATION / "qgain_v410_numerical_equivalence_to_v401")

save_table(recording_table, TABLES / "qgain_v410_recording_features")
save_table(measurement_long_frame(recording_table), TABLES / "qgain_v410_measurements_long")
save_table(model_interface_frame(recording_table), TABLES / "qgain_v410_model_interface")

# Copy immutable lower-level ledgers without recomputation, under final versioned names.
for source_name, target_name in [
    ("qgain_v401_frame_ledger.csv", "qgain_v410_frame_ledger.csv"),
    ("qgain_v401_frame_ledger.parquet", "qgain_v410_frame_ledger.parquet"),
    ("qgain_v401_segment_ledger.csv", "qgain_v410_segment_ledger.csv"),
    ("qgain_v401_segment_ledger.parquet", "qgain_v410_segment_ledger.parquet"),
    ("qgain_v401_parameters.csv", "qgain_v410_parameters.csv"),
    ("qgain_v401_parameters.parquet", "qgain_v410_parameters.parquet"),
]:
    source = SOURCE_ROOT / "tables" / source_name
    if source.exists():
        shutil.copy2(source, TABLES / target_name)

# Preserve all source validation evidence in a pinned subdirectory.
for source in (SOURCE_ROOT / "validation").glob("*"):
    if source.is_file():
        shutil.copy2(source, SOURCE_EVIDENCE / source.name)
for source_dir, target_dir in [
    (SOURCE_ROOT / "figures", FIGURES),
    (SOURCE_ROOT / "galleries", GALLERIES),
    (SOURCE_ROOT / "audit", AUDIT),
]:
    if source_dir.exists():
        shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)

g2 = pd.DataFrame([
    {"gate":"G2", "check":"all four feature values are bitwise identical to validated v4.0.1", "passed":equivalence["bitwise_equal_with_nan"].all()},
    {"gate":"G2", "check":"recording identities remain unique", "passed":not recording_table["logical_recording_id"].astype(str).duplicated().any()},
    {"gate":"G2", "check":"all retained measurements preserve missingness and support metadata", "passed":all(f"{feature}_available" in recording_table for feature in ANALYSIS_FEATURES)},
    {"gate":"G2", "check":"drift uncertainty/evidence fields are exported", "passed":all(column in recording_table for column in ["qgain_signed_drift_ci95_low_db_per_min", "qgain_signed_drift_ci95_high_db_per_min", "qgain_drift_ci_excludes_zero", "qgain_drift_evidence_status"])},
])
save_table(g2, VALIDATION / "qgain_v410_g2_checks")
display(equivalence)
display(g2)

## 3. G3–G5 — inherited estimator validation, pinned by source hashes

In [ ]:
source_gate_summary = read_table("validation/qgain_v401_gate_summary")
inherited_rows = []
for gate in ["G3", "G4", "G5"]:
    local = source_gate_summary.loc[source_gate_summary["gate"].astype(str).eq(gate)].copy()
    passed = bool(local["passed"].fillna(False).all()) if len(local) else False
    inherited_rows.append({
        "gate": gate,
        "check": f"{gate} estimator validation inherited unchanged from hashed v4.0.1 evidence",
        "passed": passed,
        "source_check_count": int(len(local)),
        "inheritance_basis": "numerical estimators unchanged; source evidence and implementation lineage pinned",
    })
inherited = pd.DataFrame(inherited_rows)
save_table(inherited, VALIDATION / "qgain_v410_g3_g5_inheritance")
display(inherited)

## 4. G6 — quantitative robustness and role-consistent interpretation

In [ ]:
segment_deletion = pd.read_csv(required_source_artifacts["segment_deletion_csv"])
boundary = pd.read_csv(required_source_artifacts["boundary_sensitivity_csv"])

# Tolerances are role-specific interpretation limits, not clinical accept/reject thresholds.
robustness_contract = pd.DataFrame([
    {"feature":"qgain_typical_speech_level_dbfs", "expected_class":"robust", "deletion_p95_limit":0.50, "guard_p95_limit":0.50},
    {"feature":"qgain_within_segment_iqr_db", "expected_class":"robust", "deletion_p95_limit":0.75, "guard_p95_limit":0.75},
    {"feature":"qgain_between_segment_mad_db", "expected_class":"moderately_sensitive", "deletion_p95_limit":1.00, "guard_p95_limit":1.50},
    {"feature":"qgain_abs_drift_db_per_min", "expected_class":"sensitive_exploratory", "deletion_p95_limit":np.nan, "guard_p95_limit":np.nan},
])
save_table(robustness_contract, VALIDATION / "qgain_v410_g6_role_specific_robustness_contract")

robustness_rows = []
for feature in ANALYSIS_FEATURES:
    deletion_local = segment_deletion.loc[segment_deletion["feature"].eq(feature)]
    boundary_local = boundary.loc[boundary["feature"].eq(feature)]
    contract = robustness_contract.set_index("feature").loc[feature]
    deletion_values = pd.to_numeric(deletion_local["absolute_change"], errors="coerce")
    boundary_values = pd.to_numeric(boundary_local["absolute_change"], errors="coerce")
    deletion_p95 = float(deletion_values.quantile(.95))
    guard_p95 = float(boundary_values.quantile(.95))
    if contract["expected_class"] == "sensitive_exploratory":
        role_consistent = (
            feature in EXPLORATORY_FEATURES
            and not bool(registry.set_index("feature").loc[feature, "standalone_gate_allowed"])
        )
        interpretation = "sensitive; retained only as exploratory/contextual"
    else:
        role_consistent = (
            deletion_p95 <= float(contract["deletion_p95_limit"])
            and guard_p95 <= float(contract["guard_p95_limit"])
        )
        interpretation = str(contract["expected_class"])
    robustness_rows.append({
        "feature": feature,
        "expected_class": contract["expected_class"],
        "segment_deletion_n": int(deletion_values.notna().sum()),
        "segment_deletion_median_abs_change": float(deletion_values.median()),
        "segment_deletion_p95_abs_change": deletion_p95,
        "segment_deletion_max_abs_change": float(deletion_values.max()),
        "segment_deletion_availability_change_fraction": float(deletion_local["availability_changed"].astype(bool).mean()),
        "boundary_n": int(boundary_values.notna().sum()),
        "boundary_median_abs_change": float(boundary_values.median()),
        "boundary_p95_abs_change": guard_p95,
        "boundary_max_abs_change": float(boundary_values.max()),
        "boundary_availability_change_fraction": float(boundary_local["availability_changed"].astype(bool).mean()),
        "role_consistent": bool(role_consistent),
        "final_interpretation": interpretation,
    })
robustness_summary = pd.DataFrame(robustness_rows)
save_table(robustness_summary, VALIDATION / "qgain_v410_g6_quantitative_robustness_summary")

g6 = pd.DataFrame([
    {"gate":"G6", "check":"quantitative segment-deletion and guard summaries exist for all features", "passed":set(robustness_summary["feature"]) == set(ANALYSIS_FEATURES)},
    {"gate":"G6", "check":"robust/contextual and secondary features satisfy role-specific limits", "passed":robustness_summary["role_consistent"].all()},
    {"gate":"G6", "check":"robustness perturbations do not materially change availability", "passed":robustness_summary[["segment_deletion_availability_change_fraction", "boundary_availability_change_fraction"]].max().max() <= 0.01},
    {"gate":"G6", "check":"drift sensitivity is explicitly quarantined as exploratory/non-gating", "passed":role_map["qgain_abs_drift_db_per_min"] == "exploratory" and not bool(registry.set_index("feature").loc["qgain_abs_drift_db_per_min", "standalone_gate_allowed"])},
])
save_table(g6, VALIDATION / "qgain_v410_g6_checks")
display(robustness_summary)
display(g6)

## 5. G7 — empirical plausibility and participant-balanced sensitivity

In [ ]:
empirical_rows = []
for feature in ANALYSIS_FEATURES:
    values = pd.to_numeric(recording_table[feature], errors="coerce")
    empirical_rows.append({
        "feature": feature,
        "recording_count": int(len(values)),
        "available_count": int(values.notna().sum()),
        "available_fraction": float(values.notna().mean()),
        "median": float(values.median()),
        "q25": float(values.quantile(.25)),
        "q75": float(values.quantile(.75)),
        "p01": float(values.quantile(.01)),
        "p99": float(values.quantile(.99)),
        "zero_fraction": float(values.eq(0).mean()),
    })
empirical = pd.DataFrame(empirical_rows)
save_table(empirical, TABLES / "qgain_v410_empirical_summary")

subject_column = "subject" if "subject" in recording_table else None
if subject_column is None:
    raise ValueError("Validated QGAIN table does not contain a subject identity column.")
subject_frame = recording_table.loc[recording_table[subject_column].notna()].copy()
subject_frame[subject_column] = subject_frame[subject_column].astype(str)
subject_frame["_date"] = pd.to_datetime(subject_frame.get("recording_date_analysis"), errors="coerce")
subject_frame = subject_frame.sort_values([subject_column, "_date", "logical_recording_id"], kind="mergesort")

deterministic = subject_frame.groupby(subject_column, as_index=False, sort=True).first()
deterministic_rows = []
for stratum, local in [("all", deterministic), *[(str(name), group) for name, group in deterministic.groupby("diagnosis_analysis", dropna=False)]]:
    for feature in ANALYSIS_FEATURES:
        values = pd.to_numeric(local[feature], errors="coerce")
        deterministic_rows.append({
            "stratum": stratum,
            "feature": feature,
            "participant_count": int(local[subject_column].nunique()),
            "available_count": int(values.notna().sum()),
            "median": float(values.median()),
            "q25": float(values.quantile(.25)),
            "q75": float(values.quantile(.75)),
        })
deterministic_summary = pd.DataFrame(deterministic_rows)
save_table(deterministic_summary, TABLES / "qgain_v410_subject_balanced_deterministic_summary")

rng = np.random.default_rng(SUBJECT_BALANCED_SEED)
resampling_rows = []
strata = [("all", subject_frame)] + [(str(name), group.copy()) for name, group in subject_frame.groupby("diagnosis_analysis", dropna=False)]
for stratum, local in strata:
    groups = [indices.to_numpy() for _, indices in local.groupby(subject_column, sort=True).groups.items()]
    for iteration in range(SUBJECT_BALANCED_ITERATIONS):
        chosen = [rng.choice(indices) for indices in groups]
        sample = local.loc[chosen]
        for feature in ANALYSIS_FEATURES:
            values = pd.to_numeric(sample[feature], errors="coerce")
            resampling_rows.append({
                "stratum": stratum,
                "iteration": iteration,
                "feature": feature,
                "participant_count": int(len(groups)),
                "available_count": int(values.notna().sum()),
                "median": float(values.median()),
                "q25": float(values.quantile(.25)),
                "q75": float(values.quantile(.75)),
            })
subject_resampling = pd.DataFrame(resampling_rows)
save_table(subject_resampling, TABLES / "qgain_v410_subject_balanced_resampling_ledger")

summary_rows = []
for (stratum, feature), local in subject_resampling.groupby(["stratum", "feature"], sort=True):
    full_local = recording_table if stratum == "all" else recording_table.loc[recording_table["diagnosis_analysis"].astype(str).eq(stratum)]
    full_values = pd.to_numeric(full_local[feature], errors="coerce")
    summary_rows.append({
        "stratum": stratum,
        "feature": feature,
        "participant_count": int(local["participant_count"].iloc[0]),
        "iterations": int(local["iteration"].nunique()),
        "recording_weighted_median": float(full_values.median()),
        "subject_balanced_median_of_medians": float(local["median"].median()),
        "subject_balanced_median_ci025": float(local["median"].quantile(.025)),
        "subject_balanced_median_ci975": float(local["median"].quantile(.975)),
        "absolute_weighting_delta": abs(float(local["median"].median()) - float(full_values.median())),
    })
subject_balance_summary = pd.DataFrame(summary_rows)
save_table(subject_balance_summary, TABLES / "qgain_v410_subject_balanced_resampling_summary")

g7 = pd.DataFrame([
    {"gate":"G7", "check":"all empirical distributions summarized", "passed":len(empirical) == 4 and empirical["available_count"].gt(0).all()},
    {"gate":"G7", "check":"one-recording-per-participant deterministic summary produced", "passed":deterministic[subject_column].nunique() == subject_frame[subject_column].nunique()},
    {"gate":"G7", "check":"participant-balanced resampling completed", "passed":subject_resampling["iteration"].nunique() == SUBJECT_BALANCED_ITERATIONS and set(subject_resampling["feature"]) == set(ANALYSIS_FEATURES)},
    {"gate":"G7", "check":"label-blind empirical galleries preserved without source errors", "passed":source_gallery_errors.empty and (GALLERIES / "qgain_gallery_index.csv").exists()},
])
save_table(g7, VALIDATION / "qgain_v410_g7_checks")
display(empirical)
display(subject_balance_summary)
display(g7)

## 6. G8 — persistence, redundancy, drift evidence, and ML contract

In [ ]:
persistence = pd.read_csv(required_source_artifacts["persistence_csv"])
drift_permutation = pd.read_csv(required_source_artifacts["drift_permutation_csv"])
save_table(persistence, VALIDATION / "qgain_v410_repeated_recording_persistence")
save_table(drift_permutation, VALIDATION / "qgain_v410_drift_time_order_permutation")

correlations = recording_table[list(ANALYSIS_FEATURES)].corr(method="spearman")
correlations.to_csv(TABLES / "qgain_v410_spearman_correlations.csv")
upper = correlations.where(np.triu(np.ones(correlations.shape), k=1).astype(bool)).stack()
max_abs_correlation = float(upper.abs().max()) if len(upper) else 0.0

model_ready = model_interface_frame(recording_table)
long_measurements = measurement_long_frame(recording_table)

# Drift evidence is descriptive and does not convert drift into a validated acquisition detector.
drift_ci_fraction = float(recording_table["qgain_drift_ci_excludes_zero"].astype(bool).mean())
permutation_significant_fraction = float((pd.to_numeric(drift_permutation["permutation_pvalue"], errors="coerce") < .05).mean())
drift_evidence_summary = pd.DataFrame([{
    "recording_count": int(len(recording_table)),
    "ci_excludes_zero_count": int(recording_table["qgain_drift_ci_excludes_zero"].astype(bool).sum()),
    "ci_excludes_zero_fraction": drift_ci_fraction,
    "permutation_audit_count": int(len(drift_permutation)),
    "permutation_p_lt_0_05_count": int((pd.to_numeric(drift_permutation["permutation_pvalue"], errors="coerce") < .05).sum()),
    "permutation_p_lt_0_05_fraction": permutation_significant_fraction,
    "interpretation": "weak task-level trend evidence; drift retained as exploratory/contextual only",
}])
save_table(drift_evidence_summary, VALIDATION / "qgain_v410_drift_evidence_summary")

g8 = pd.DataFrame([
    {"gate":"G8", "check":"no exact or near-duplicate retained feature pair", "passed":max_abs_correlation < 0.90, "maximum_absolute_spearman":max_abs_correlation},
    {"gate":"G8", "check":"repeated-recording persistence quantified for all features", "passed":set(persistence["feature"]) == set(ANALYSIS_FEATURES) and persistence["first_second_spearman"].notna().all()},
    {"gate":"G8", "check":"ML interface contains every value, availability mask, status, and support tier", "passed":all(all(column in model_ready for column in [feature, f"{feature}_available", f"{feature}_status", f"{feature}_support_tier"]) for feature in ANALYSIS_FEATURES)},
    {"gate":"G8", "check":"drift CI and time-order evidence are exported and explicitly exploratory", "passed":"qgain_drift_evidence_status" in model_ready and role_map["qgain_abs_drift_db_per_min"] == "exploratory"},
    {"gate":"G8", "check":"no QGAIN scalar or standalone threshold is exposed", "passed":not model_ready["qgain_family_scalar_available"].astype(bool).any() and model_ready["qgain_family_scalar_status"].astype(str).eq("prohibited_not_constructed").all() and not model_ready["qgain_standalone_reject_allowed"].astype(bool).any()},
    {"gate":"G8", "check":"optional ITU-T P.56 comparability analysis", "passed":False, "status":"optional_not_blocking"},
])
save_table(g8, VALIDATION / "qgain_v410_g8_checks")
display(persistence)
display(drift_evidence_summary)
display(g8)

## 7. G9 — event adjudication

In [ ]:
g9 = pd.DataFrame([{
    "gate":"G9",
    "check":"event adjudication",
    "passed":True,
    "status":"not_applicable_no_event_detector",
    "note":"The rejected sustained level-step detector is absent from the final module, registry, tables, and ML interface.",
}])
save_table(g9, VALIDATION / "qgain_v410_g9_checks")
display(g9)

## 8. G10 — final decisions and freeze-readiness manifest

In [ ]:
feature_decisions = pd.DataFrame([
    {
        "feature":"qgain_typical_speech_level_dbfs",
        "final_decision":"RETAIN_CONTEXTUAL",
        "publication_role":"contextual measurement",
        "default_manuscript_inclusion":True,
        "ml_interface":True,
        "standalone_gate_allowed":False,
        "confidence":"high",
        "publication_claim":"analysis-view operating-level context only; not SPL, vocal intensity, gain, AGC, or P.56",
    },
    {
        "feature":"qgain_within_segment_iqr_db",
        "final_decision":"RETAIN_PRIMARY_MIXED",
        "publication_role":"primary mixed acquisition/physiology descriptor",
        "default_manuscript_inclusion":True,
        "ml_interface":True,
        "standalone_gate_allowed":False,
        "confidence":"high_with_strong_claim_boundary",
        "publication_claim":"within-segment recorded-level dispersion; compatible with modulation but not AGC-specific",
    },
    {
        "feature":"qgain_between_segment_mad_db",
        "final_decision":"RETAIN_SECONDARY_MIXED",
        "publication_role":"secondary segment-level descriptor",
        "default_manuscript_inclusion":False,
        "ml_interface":True,
        "standalone_gate_allowed":False,
        "confidence":"moderate",
        "publication_claim":"between-segment recorded-level dispersion; task- and boundary-dependent",
    },
    {
        "feature":"qgain_abs_drift_db_per_min",
        "final_decision":"RETAIN_EXPLORATORY_CONTEXTUAL",
        "publication_role":"exploratory contextual trend descriptor",
        "default_manuscript_inclusion":False,
        "ml_interface":True,
        "standalone_gate_allowed":False,
        "confidence":"moderate_estimator_low_acquisition_specificity",
        "publication_claim":"absolute ordered level trend with CI; exploratory and not acquisition-specific",
    },
])
save_table(feature_decisions, VALIDATION / "qgain_v410_g10_feature_decisions")

review_accepted = SCIENTIFIC_REVIEW_DECISION == "ACCEPT_QGAIN_V410"
g10 = pd.DataFrame([
    {"gate":"G10", "check":"exact v4.1.0 scientific acceptance token", "passed":review_accepted},
    {"gate":"G10", "check":"scientific reviewer identified", "passed":bool(SCIENTIFIC_REVIEWER.strip())},
    {"gate":"G10", "check":"scientific rationale recorded", "passed":len(SCIENTIFIC_REVIEW_RATIONALE.strip()) >= 80},
    {"gate":"G10", "check":"all retained features have explicit final decisions", "passed":set(feature_decisions["feature"]) == set(ANALYSIS_FEATURES)},
    {"gate":"G10", "check":"no feature is authorized as a standalone gate", "passed":not feature_decisions["standalone_gate_allowed"].any()},
])
save_table(g10, VALIDATION / "qgain_v410_g10_checks")

all_checks = pd.concat([g1, g2, inherited, g6, g7, g8, g9, g10], ignore_index=True, sort=False)
all_checks["blocking"] = all_checks["gate"].isin(["G1","G2","G3","G4","G5","G6","G7","G8","G10"])
all_checks.loc[all_checks["check"].astype(str).eq("optional ITU-T P.56 comparability analysis"), "blocking"] = False
save_table(all_checks, VALIDATION / "qgain_v410_gate_summary")
blocking_pass = bool(all_checks.loc[all_checks["blocking"], "passed"].fillna(False).all())

manifest = {
    "measurement_version": MEASUREMENT_VERSION,
    "family": "QGAIN",
    "family_display_name": "Recorded level and level dynamics",
    "freeze_status": "ready_for_atomic_freeze" if blocking_pass else "blocked",
    "candidate_only": True,
    "freeze_allowed": blocking_pass,
    "source_measurement_version": source_manifest.get("measurement_version"),
    "source_cohort_extraction_completed": bool(source_manifest.get("cohort_extraction_completed")),
    "source_artifact_inventory_sha256": sha256((VALIDATION / "qgain_v410_source_artifact_inventory.csv")),
    "numerical_equivalence_to_v401": bool(equivalence["bitwise_equal_with_nan"].all()),
    "scientific_review_decision": SCIENTIFIC_REVIEW_DECISION,
    "scientific_reviewer": SCIENTIFIC_REVIEWER,
    "scientific_review_rationale": SCIENTIFIC_REVIEW_RATIONALE,
    "analysis_features": list(ANALYSIS_FEATURES),
    "primary_analysis_features": list(PRIMARY_ANALYSIS_FEATURES),
    "secondary_analysis_features": list(SECONDARY_ANALYSIS_FEATURES),
    "exploratory_features": list(EXPLORATORY_FEATURES),
    "recording_count": int(len(recording_table)),
    "participant_count": int(subject_frame[subject_column].nunique()),
    "standalone_reject_allowed": False,
    "family_scalar_status": "prohibited_not_constructed",
    "decision_threshold_status": "not_calibrated",
    "p56_comparator_status": "optional_not_blocking",
    "executed_notebook_required_for_freeze": True,
    "atomic_freeze_script_required": True,
}
manifest_path = MANIFESTS / "qgain_v410_candidate_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

display(feature_decisions)
display(g10)
print(json.dumps(manifest, indent=2))
if not blocking_pass:
    failed = all_checks.loc[all_checks["blocking"] & ~all_checks["passed"].fillna(False), ["gate","check","status"]]
    display(failed)
    raise RuntimeError("QGAIN v4.1.0 freeze candidate is blocked by one or more gates.")

## 9. Feature passports

In [ ]:
registry_index = registry.set_index("feature")
empirical_index = empirical.set_index("feature")
robustness_index = robustness_summary.set_index("feature")
persistence_index = persistence.set_index("feature")
decision_index = feature_decisions.set_index("feature")

passport_index_rows = []
for feature in ANALYSIS_FEATURES:
    payload = {
        "feature": feature,
        "measurement_version": MEASUREMENT_VERSION,
        "family": "QGAIN",
        "definition": registry_index.loc[feature].to_dict(),
        "final_decision": decision_index.loc[feature].to_dict(),
        "empirical_summary": empirical_index.loc[feature].to_dict(),
        "robustness_summary": robustness_index.loc[feature].to_dict(),
        "persistence_summary": persistence_index.loc[feature].to_dict(),
        "standalone_gate_allowed": False,
        "family_scalar_membership": "none_prohibited",
    }
    json_path = PASSPORTS / f"{feature}_passport.json"
    json_path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
    markdown = f"""# {feature}\n\n**Version:** {MEASUREMENT_VERSION}  \n**Final decision:** {payload['final_decision']['final_decision']}  \n**Publication role:** {payload['final_decision']['publication_role']}  \n**Unit:** {payload['definition']['unit']}  \n**Estimator:** {payload['definition']['estimator']}  \n**Estimand:** {payload['definition']['estimand']}  \n**Claim limit:** {payload['definition']['claim_limit']}  \n**Known confounds:** {payload['definition']['known_confounds']}  \n**Minimum support:** {payload['definition']['minimum_support']}  \n**Robustness class:** {payload['definition']['robustness_class']}  \n**Standalone gate:** prohibited  \n**Family scalar:** prohibited\n\n## Empirical summary\n\n- Available: {payload['empirical_summary']['available_count']} / {payload['empirical_summary']['recording_count']}\n- Median: {payload['empirical_summary']['median']:.6g} {payload['definition']['unit']}\n- IQR: [{payload['empirical_summary']['q25']:.6g}, {payload['empirical_summary']['q75']:.6g}]\n\n## Robustness\n\n- Segment-deletion p95 absolute change: {payload['robustness_summary']['segment_deletion_p95_abs_change']:.6g}\n- Boundary-guard p95 absolute change: {payload['robustness_summary']['boundary_p95_abs_change']:.6g}\n- Final interpretation: {payload['robustness_summary']['final_interpretation']}\n\n## Repeated-recording persistence\n\n- First–second Spearman: {payload['persistence_summary']['first_second_spearman']:.6g}\n- ICC(1,1), first two: {payload['persistence_summary']['icc_1_1_first_two']:.6g}\n\n## Permitted use\n\n{payload['final_decision']['publication_claim']}\n"""
    md_path = PASSPORTS / f"{feature}_passport.md"
    md_path.write_text(markdown, encoding="utf-8")
    passport_index_rows.append({"feature":feature, "json_path":str(json_path), "markdown_path":str(md_path)})
passport_index = pd.DataFrame(passport_index_rows)
save_table(passport_index, PASSPORTS / "qgain_v410_feature_passport_index")
display(passport_index)

## 10. Package tests and completion

In [ ]:
if RUN_PACKAGE_TESTS:
    completed = subprocess.run(
        [sys.executable, "-m", "pytest", str(PROJECT_ROOT / "tests" / "test_qgain_v410.py"), "-q"],
        cwd=PROJECT_ROOT,
        check=False,
        capture_output=True,
        text=True,
    )
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if completed.returncode:
        raise RuntimeError("QGAIN v4.1.0 package tests failed.")

print("QGAIN v4.1.0 FINALIZATION COMPLETE")
print(f"Freeze-ready candidate: {OUTPUT_ROOT}")
print("Save this executed notebook, close JupyterLab, then run scripts/freeze_qgain_v410.ps1.")